# P1-E16.7H-2 Renamed (P1-E11.H_2)FINAL Frozen External Compatibility, Domain Shift, Calibration, and Probability-Quality Audit

**Correction.** This version fixes the final integrity-check variable collision in v1. The dataset–endpoint set is now retained as `observed_pairs`, while calculated metric dictionaries use `observed_metrics`. No cohort, prediction, metric definition, or scientific result was changed.

**Purpose.** Complete the reviewer-requested external compatibility and probability-quality audit using the already frozen P1-E16.7E participant-level predictions. This notebook does not train, select, recalibrate, adapt, or tune a model.

**Evidence label.** Locked external re-evaluation—not historically untouched validation—because AHEPA and Dryad/P-ADIC were inspected in earlier experiments.

**Endpoint contract.** AHEPA supports Normal vs AD only. Dryad/P-ADIC supports Normal vs AD, MCI vs AD, Normal vs MCI, and three-class Normal/MCI/AD. Results are never pooled across external datasets.


## 1. Configuration and immutable analysis contract

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, brier_score_loss,
    confusion_matrix, f1_score, log_loss, matthews_corrcoef, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED=20261012
BOOTSTRAP_REPS=5000
CALIBRATION_BINS=10
TASKS={'normal_vs_ad':['Normal','AD'],'mci_vs_ad':['MCI','AD'],
       'normal_vs_mci':['Normal','MCI'],'three_class':['Normal','MCI','AD']}
TASK_LABELS={'normal_vs_ad':'Normal vs AD','mci_vs_ad':'MCI vs AD',
             'normal_vs_mci':'Normal vs MCI','three_class':'Three class'}
EXTERNAL_PROTOCOL={'AHEPA':['normal_vs_ad'],
                   'Dryad':['normal_vs_ad','mci_vs_ad','normal_vs_mci','three_class']}
METHODS=['Frozen conventional benchmark','Endpoint-selected CEEDNet/SSL']

PROJECT_ROOT=Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT=PROJECT_ROOT/'paper1_outputs'
LEGACY_OUTPUTS_ROOT=Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E16_7E_ROOT=OUTPUTS_ROOT/'p1_e16_7e_frozen_external_reevaluation_transportability_shift_chronology'
E16_7BR_ROOT=OUTPUTS_ROOT/'p1_e16_7b_r_resume_safe_unified_four_endpoint_nested_method_comparison'
OUTPUT_ROOT=OUTPUTS_ROOT/'p1_e16_7h_2_external_compatibility_domain_shift_calibration_probability_quality'
TABLES=OUTPUT_ROOT/'tables'; FIGURES=OUTPUT_ROOT/'figures'
for folder in (OUTPUT_ROOT,TABLES,FIGURES): folder.mkdir(parents=True,exist_ok=True)

# Set an exact path here only if automatic resolution reports ambiguity.
EXACT_EXTERNAL_PREDICTIONS=None
EXACT_DEVELOPMENT_PREDICTIONS=None
EXACT_DEVELOPMENT_FEATURES=None
EXACT_AHEPA_FEATURES=None
EXACT_DRYAD_FEATURES=None

def sha256_file(path,block=1<<20):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(block),b''): h.update(chunk)
    return h.hexdigest()

def save_table(frame,name):
    path=TABLES/f'{name}.csv'; frame.to_csv(path,index=False); return path

def save_figure(fig,name):
    fig.tight_layout(); fig.savefig(FIGURES/f'{name}.pdf',bbox_inches='tight')
    fig.savefig(FIGURES/f'{name}.png',dpi=500,bbox_inches='tight',facecolor='white')

def unique_by_content(paths,label):
    paths=sorted({Path(p) for p in paths if Path(p).is_file()})
    if not paths: raise FileNotFoundError(f'{label}: no candidate file was found. Set its EXACT_* path in Cell 2.')
    families={}
    for path in paths: families.setdefault(sha256_file(path),[]).append(path)
    if len(families)!=1:
        display(pd.DataFrame([{'path':str(p),'sha256':sha256_file(p)} for p in paths]))
        raise RuntimeError(f'{label}: {len(families)} different content families were found. Set the exact path in Cell 2.')
    return sorted(next(iter(families.values())),key=lambda p:(len(str(p)),str(p)))[0]

def resolve(exact,roots,patterns,label):
    if exact:
        path=Path(exact)
        if not path.is_file(): raise FileNotFoundError(path)
        return path
    found=[]
    for root in roots:
        root=Path(root)
        if root.is_dir():
            for pattern in patterns: found.extend(root.rglob(pattern))
    return unique_by_content(found,label)

sns.set_theme(style='whitegrid',context='paper',font_scale=1.05)
plt.rcParams.update({'figure.dpi':130,'savefig.dpi':500,'pdf.fonttype':42,'ps.fonttype':42})
rng=np.random.default_rng(SEED)
print({'output':str(OUTPUT_ROOT),'bootstrap_repetitions':BOOTSTRAP_REPS,
       'external_evidence':'locked re-evaluation; no external model modification'})

## 2. Authenticate and reconcile the saved external predictions

In [ ]:
external_predictions_path=resolve(
    EXACT_EXTERNAL_PREDICTIONS,[E16_7E_ROOT],
    ['07_MATCHED_FROZEN_EXTERNAL_PARTICIPANT_PREDICTIONS.csv'],
    'P1-E16.7E matched external predictions')

external=pd.read_csv(external_predictions_path,dtype={'participant_id':str})
required={'dataset','task','method','participant_id','true_label','predicted_label'}
if not required.issubset(external): raise KeyError(f'External predictions lack: {sorted(required-set(external.columns))}')
external['dataset']=external.dataset.replace({'Dryad/P-ADIC':'Dryad','DRYAD':'Dryad'})
expected={(d,t) for d,tasks in EXTERNAL_PROTOCOL.items() for t in tasks}
observed_pairs=set(map(tuple,external[['dataset','task']].drop_duplicates().to_numpy()))
if observed_pairs!=expected: raise AssertionError(f'External endpoint contract mismatch. Missing={expected-observed_pairs}; unexpected={observed_pairs-expected}')
if set(external.method)!=set(METHODS): raise AssertionError(f'Method contract mismatch: {sorted(external.method.unique())}')
if external.duplicated(['dataset','task','method','participant_id']).any(): raise AssertionError('Duplicate participant prediction detected.')

compatibility=[]
for (dataset,task,method),q in external.groupby(['dataset','task','method'],sort=False):
    classes=TASKS[task]; probability_columns=[f'probability_{c}' for c in classes]
    if not set(probability_columns).issubset(q): raise KeyError(f'{dataset}/{task}: missing probability columns.')
    P=q[probability_columns].to_numpy(float)
    compatibility.extend([
      {'dataset':dataset,'endpoint':task,'method':method,'check':'required classes present','pass':set(q.true_label)==set(classes),'detail':str(q.true_label.value_counts().to_dict())},
      {'dataset':dataset,'endpoint':task,'method':method,'check':'one row per participant','pass':not q.participant_id.duplicated().any(),'detail':len(q)},
      {'dataset':dataset,'endpoint':task,'method':method,'check':'finite normalized probabilities','pass':np.isfinite(P).all() and np.allclose(P.sum(1),1,atol=1e-5),'detail':f'n={len(q)}'},
      {'dataset':dataset,'endpoint':task,'method':method,'check':'labels and predictions in class contract','pass':set(q.true_label).issubset(classes) and set(q.predicted_label).issubset(classes),'detail':str(classes)}])
compatibility=pd.DataFrame(compatibility)
if not compatibility['pass'].all(): display(compatibility); raise AssertionError('External compatibility audit failed.')
source_manifest=pd.DataFrame([{'role':'matched frozen external participant predictions','path':str(external_predictions_path),'sha256':sha256_file(external_predictions_path)}])
save_table(source_manifest,'01_INPUT_PROVENANCE'); save_table(compatibility,'02_EXTERNAL_COMPATIBILITY_AUDIT')
display(external.groupby(['dataset','task','true_label']).participant_id.nunique().rename('participants').reset_index().drop_duplicates())
display(compatibility)

## 3. Probability-quality definitions and calibration-bin construction

In [ ]:
def wilson_interval(successes,n,z=1.959963984540054):
    if n==0: return (np.nan,np.nan)
    p=successes/n; den=1+z*z/n; centre=(p+z*z/(2*n))/den
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return max(0,centre-half),min(1,centre+half)

def classwise_ece(y,P,n_bins=10):
    edges=np.linspace(0,1,n_bins+1); values=[]
    for k in range(P.shape[1]):
        observed=(y==k).astype(int); ece=0.0
        for b,(lo,hi) in enumerate(zip(edges[:-1],edges[1:])):
            use=(P[:,k]>=lo)&(P[:,k]<(hi if b<n_bins-1 else hi+1e-12))
            if use.any(): ece+=use.mean()*abs(observed[use].mean()-P[use,k].mean())
        values.append(ece)
    return float(np.mean(values))

def evaluate_probability_quality(q,task):
    classes=TASKS[task]; c2i={c:i for i,c in enumerate(classes)}
    y=q.true_label.map(c2i).to_numpy(int); yp=q.predicted_label.map(c2i).to_numpy(int)
    P=np.clip(q[[f'probability_{c}' for c in classes]].to_numpy(float),1e-15,1-1e-15)
    P=P/P.sum(1,keepdims=True); onehot=np.eye(len(classes))[y]
    prevalence=np.bincount(y,minlength=len(classes))/len(y)
    brier=float(np.mean(np.sum((P-onehot)**2,axis=1)))
    baseline=float(np.mean(np.sum((prevalence[None,:]-onehot)**2,axis=1)))
    out={'n':len(q),'accuracy':accuracy_score(y,yp),'balanced_accuracy':balanced_accuracy_score(y,yp),
         'macro_f1':f1_score(y,yp,average='macro',zero_division=0),'mcc':matthews_corrcoef(y,yp),
         'log_loss':log_loss(y,P,labels=np.arange(len(classes))),
         'multiclass_brier':brier,'prevalence_baseline_brier':baseline,
         'brier_skill_score':1-brier/baseline if baseline>0 else np.nan,
         'classwise_ece_10bin':classwise_ece(y,P,CALIBRATION_BINS)}
    if len(classes)==2:
        tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
        out.update({'sensitivity':tp/(tp+fn),'specificity':tn/(tn+fp),
                    'roc_auc':roc_auc_score(y,P[:,1]),'binary_brier':brier_score_loss(y,P[:,1])})
    else:
        out['roc_auc']=roc_auc_score(y,P,multi_class='ovr',average='macro',labels=np.arange(len(classes)))
    return out

def calibration_rows(q,dataset,task,method):
    classes=TASKS[task]; c2i={c:i for i,c in enumerate(classes)}
    y=q.true_label.map(c2i).to_numpy(int); P=q[[f'probability_{c}' for c in classes]].to_numpy(float)
    P=np.clip(P,0,1); P=P/P.sum(1,keepdims=True); rows=[]; edges=np.linspace(0,1,CALIBRATION_BINS+1)
    for k,label in enumerate(classes):
        observed=(y==k).astype(int)
        for b,(lo,hi) in enumerate(zip(edges[:-1],edges[1:])):
            use=(P[:,k]>=lo)&(P[:,k]<(hi if b<CALIBRATION_BINS-1 else hi+1e-12)); n=int(use.sum())
            if not n: continue
            successes=int(observed[use].sum()); low,high=wilson_interval(successes,n)
            rows.append({'dataset':dataset,'task':task,'method':method,'class':label,'bin':b+1,'n':n,
                         'mean_predicted_probability':P[use,k].mean(),'observed_fraction':successes/n,
                         'wilson95_low':low,'wilson95_high':high})
    return rows

## 4. Compute external metrics, bootstrap intervals, and Brier skill

In [ ]:
metric_rows=[]; interval_rows=[]; calibration=[]
for (dataset,task,method),q in external.groupby(['dataset','task','method'],sort=False):
    q=q.reset_index(drop=True); classes=TASKS[task]; observed_metrics=evaluate_probability_quality(q,task)
    metric_rows.append({'dataset':dataset,'task':task,'method':method,**observed_metrics})
    calibration.extend(calibration_rows(q,dataset,task,method))
    strata={c:np.flatnonzero(q.true_label.to_numpy()==c) for c in classes}; boot=[]
    for _ in range(BOOTSTRAP_REPS):
        index=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes])
        boot.append(evaluate_probability_quality(q.iloc[index],task))
    boot=pd.DataFrame(boot)
    for metric,estimate in observed_metrics.items():
        if metric!='n':
            interval_rows.append({'dataset':dataset,'task':task,'method':method,'metric':metric,
              'estimate':estimate,'ci95_low':boot[metric].quantile(.025),'ci95_high':boot[metric].quantile(.975)})

metrics=pd.DataFrame(metric_rows); intervals=pd.DataFrame(interval_rows); calibration=pd.DataFrame(calibration)
save_table(metrics,'03_EXTERNAL_DISCRIMINATION_AND_PROBABILITY_QUALITY')
save_table(intervals,'04_PARTICIPANT_STRATIFIED_BOOTSTRAP_INTERVALS')
save_table(calibration,'05_CALIBRATION_BINS_WITH_WILSON_INTERVALS')
display(metrics.round(4))

## 5. Optional matched internal–external probability-quality comparison

In [ ]:
development_path=None
try:
    development_path=resolve(EXACT_DEVELOPMENT_PREDICTIONS,[E16_7BR_ROOT,OUTPUTS_ROOT],
      ['*OOF*PREDICTIONS*.csv','*NESTED*PREDICTIONS*.csv'],
      'development out-of-fold predictions')
except (FileNotFoundError,RuntimeError) as exc:
    warnings.warn(f'Internal comparison skipped until an exact development OOF table is set: {exc}')

internal_external=pd.DataFrame()
if development_path is not None:
    development=pd.read_csv(development_path,dtype={'recording_id':str,'participant_id':str})
    rename={c:'task' for c in ['endpoint'] if c in development and 'task' not in development}
    rename.update({c:'true_label' for c in ['label','y_true'] if c in development and 'true_label' not in development})
    rename.update({c:'predicted_label' for c in ['prediction','y_pred'] if c in development and 'predicted_label' not in development})
    development=development.rename(columns=rename)
    needed={'task','method','true_label','predicted_label'}
    if not needed.issubset(development):
        warnings.warn(f'Development table cannot be compared; missing {sorted(needed-set(development.columns))}.')
    else:
        rows=[]
        for (task,method),q in development.groupby(['task','method']):
            if task not in TASKS or not set(q.true_label.dropna()).issubset(TASKS[task]): continue
            probability_columns=[f'probability_{c}' for c in TASKS[task]]
            if not set(probability_columns).issubset(q): continue
            rows.append({'dataset':'CAUEEG development OOF','task':task,'method':method,**evaluate_probability_quality(q,task)})
        internal_external=pd.concat([pd.DataFrame(rows),metrics],ignore_index=True,sort=False)
        save_table(internal_external,'06_INTERNAL_OOF_VERSUS_EXTERNAL_PROBABILITY_QUALITY')
        source_manifest.loc[len(source_manifest)]={'role':'development out-of-fold predictions','path':str(development_path),'sha256':sha256_file(development_path)}
display(internal_external.round(4) if len(internal_external) else pd.DataFrame({'note':['Set EXACT_DEVELOPMENT_PREDICTIONS to enable the matched internal–external comparison.']}))

## 6. Feature shift and domain separability (diagnostic only)

In [ ]:
feature_specs={
 'CAUEEG':(EXACT_DEVELOPMENT_FEATURES,[OUTPUTS_ROOT,LEGACY_OUTPUTS_ROOT],['*LOCKED*RECORDING*FEATURE*MATRIX*.npz','*LOCKED*FEATURE*MATRIX*.npz']),
 'AHEPA':(EXACT_AHEPA_FEATURES,[LEGACY_OUTPUTS_ROOT/'p1_e10_2_ahepa_frozen_normal_ad_external_validation'],['AHEPA_LOCKED_NONSPIKING_FEATURE_MATRIX.npz']),
 'Dryad':(EXACT_DRYAD_FEATURES,[LEGACY_OUTPUTS_ROOT/'p1_e10_3_dryad_frozen_three_endpoint_external_validation'],['DRYAD_LOCKED_NONSPIKING_FEATURE_MATRIX.npz'])}
feature_data={}; feature_notes=[]
for dataset,(exact,roots,patterns) in feature_specs.items():
    try:
        path=resolve(exact,roots,patterns,f'{dataset} feature matrix')
        with np.load(path,allow_pickle=False) as z:
            X=np.asarray(z['X'],float); names=z['feature_names'].astype(str)
        if X.ndim!=2 or X.shape[1]!=len(names) or not np.isfinite(X).all(): raise AssertionError(f'{dataset}: invalid feature matrix.')
        feature_data[dataset]=(X,names,path)
        source_manifest.loc[len(source_manifest)]={'role':f'{dataset} feature matrix','path':str(path),'sha256':sha256_file(path)}
    except (FileNotFoundError,RuntimeError,AssertionError,KeyError) as exc:
        feature_notes.append({'dataset':dataset,'status':'not evaluated','reason':str(exc)})

feature_shift=[]; domain_rows=[]
if set(feature_data)=={'CAUEEG','AHEPA','Dryad'}:
    Xdev,names,_=feature_data['CAUEEG']
    for dataset in ('AHEPA','Dryad'):
        Xext,external_names,_=feature_data[dataset]
        if not np.array_equal(names,external_names): raise AssertionError(f'{dataset}: feature schema/order differs from CAUEEG.')
        pooled=np.sqrt((Xdev.var(0,ddof=1)+Xext.var(0,ddof=1))/2)
        smd=np.divide(Xext.mean(0)-Xdev.mean(0),pooled,out=np.zeros_like(pooled),where=pooled>1e-12)
        feature_shift.extend([{'dataset':dataset,'feature_name':name,'smd_external_minus_caueeg':value,'absolute_smd':abs(value)} for name,value in zip(names,smd)])
        X=np.vstack([Xdev,Xext]); y=np.r_[np.zeros(len(Xdev),int),np.ones(len(Xext),int)]
        folds=min(5,int(np.bincount(y).min())); cv=StratifiedKFold(folds,shuffle=True,random_state=SEED)
        pipeline=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler()),
                           ('model',LogisticRegression(max_iter=3000,class_weight='balanced',random_state=SEED))])
        probability=cross_val_predict(pipeline,X,y,cv=cv,method='predict_proba',n_jobs=-1)[:,1]
        domain_rows.append({'external_dataset':dataset,'n_caueeg':len(Xdev),'n_external':len(Xext),
                            'domain_classifier_roc_auc':roc_auc_score(y,probability),'cv_folds':folds})
feature_shift=pd.DataFrame(feature_shift); domain_separability=pd.DataFrame(domain_rows)
if len(feature_shift): save_table(feature_shift,'07_FEATURE_SHIFT_STANDARDIZED_MEAN_DIFFERENCES')
if len(domain_separability): save_table(domain_separability,'08_DOMAIN_CLASSIFIER_SEPARABILITY')
save_table(pd.DataFrame(feature_notes),'09_OPTIONAL_FEATURE_DIAGNOSTIC_STATUS')
display(domain_separability if len(domain_separability) else pd.DataFrame(feature_notes))

## 7. Publication-ready reliability and shift figures

In [ ]:
# Figure 1: reliability curves for every supported external dataset–endpoint pair.
panels=sorted(expected); fig,axes=plt.subplots(3,2,figsize=(11,12)); axes=axes.ravel()
colors={METHODS[0]:'#4C78A8',METHODS[1]:'#F28E2B'}; styles={'Normal':'-','MCI':'--','AD':':'}
for ax,(dataset,task) in zip(axes,panels):
    q=calibration.query('dataset==@dataset and task==@task')
    for (method,label),z in q.groupby(['method','class']):
        ax.plot(z.mean_predicted_probability,z.observed_fraction,marker='o',ms=3,
                color=colors[method],ls=styles.get(label,'-'),
                label=f"{'Conventional' if method==METHODS[0] else 'Selected deep'} · {label}")
    ax.plot([0,1],[0,1],color='0.4',ls='--',lw=1); ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_title(f'{dataset} · {TASK_LABELS[task]}'); ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed fraction')
    ax.legend(frameon=False,fontsize=6)
for ax in axes[len(panels):]: ax.axis('off')
save_figure(fig,'Figure_1_External_Reliability_Diagrams'); plt.show()

# Figure 2: Brier skill score. Zero means no improvement over empirical prevalence.
q=metrics.sort_values(['dataset','task','method']).copy(); q['label']=q.apply(lambda r:f"{r.dataset} · {TASK_LABELS[r.task]} · {'Conv.' if r.method==METHODS[0] else 'Deep'}",axis=1)
fig,ax=plt.subplots(figsize=(10,6)); colors_bar=[colors[m] for m in q.method]
ax.barh(np.arange(len(q)),q.brier_skill_score,color=colors_bar); ax.axvline(0,color='0.25',ls='--')
ax.set_yticks(np.arange(len(q))); ax.set_yticklabels(q.label); ax.set_xlabel('Brier skill score versus empirical-prevalence reference'); ax.set_title('External probability quality')
save_figure(fig,'Figure_2_External_Brier_Skill_Scores'); plt.show()

# Figure 3: feature-shift distribution, when feature inputs authenticate.
if len(feature_shift):
    fig,ax=plt.subplots(figsize=(8,5)); sns.histplot(feature_shift,x='absolute_smd',hue='dataset',element='step',bins=35,common_norm=False,ax=ax)
    ax.axvline(.5,color='#B22222',ls='--'); ax.set_xlabel('Absolute standardized mean difference'); ax.set_title('External feature shift relative to CAUEEG')
    save_figure(fig,'Figure_3_External_Feature_Shift'); plt.show()

## 8. Final integrity, chronology, and claim lock

In [ ]:
checks=[]
for dataset,tasks in EXTERNAL_PROTOCOL.items():
    for task in tasks:
        a=external.loc[(external.dataset==dataset)&(external.task==task)&(external.method==METHODS[0])].sort_values('participant_id')
        b=external.loc[(external.dataset==dataset)&(external.task==task)&(external.method==METHODS[1])].sort_values('participant_id')
        checks.extend([
          {'scope':f'{dataset}:{task}','check':'matched participants and labels across methods','pass':a.participant_id.reset_index(drop=True).equals(b.participant_id.reset_index(drop=True)) and a.true_label.reset_index(drop=True).equals(b.true_label.reset_index(drop=True))},
          {'scope':f'{dataset}:{task}','check':'all required classes present','pass':set(a.true_label)==set(TASKS[task])},
          {'scope':f'{dataset}:{task}','check':'endpoint-specific chance is mathematically correct','pass':np.isclose(1/len(TASKS[task]),.5 if len(TASKS[task])==2 else 1/3)}])
checks.extend([
 {'scope':'all','check':'exactly five compatible external dataset-endpoint combinations','pass':observed_pairs==expected},
 {'scope':'all','check':'external datasets reported separately','pass':set(external.dataset)=={'AHEPA','Dryad'}},
 {'scope':'all','check':'both frozen methods retained','pass':set(external.method)==set(METHODS)},
 {'scope':'all','check':'all probabilities finite and normalized','pass':compatibility.loc[compatibility.check.eq('finite normalized probabilities'),'pass'].all()},
 {'scope':'all','check':'participant-stratified bootstrap complete','pass':BOOTSTRAP_REPS==5000 and len(intervals)>0},
 {'scope':'all','check':'prevalence-reference Brier and skill computed','pass':metrics[['prevalence_baseline_brier','brier_skill_score']].notna().all().all()},
 {'scope':'all','check':'calibration tables and Wilson intervals complete','pass':len(calibration)>0 and calibration[['wilson95_low','wilson95_high']].notna().all().all()},
 {'scope':'all','check':'no external refitting, selection, threshold optimization, recalibration, or adaptation','pass':True},
 {'scope':'all','check':'historically untouched claim prohibited','pass':True},
 {'scope':'all','check':'no performance target used as a gate','pass':True}])
checks=pd.DataFrame(checks); save_table(checks,'10_FINAL_INTEGRITY_CHECKS'); save_table(source_manifest,'01_INPUT_PROVENANCE')
display(checks)
if not checks['pass'].all(): raise AssertionError('P1-E16.7H-2 integrity failure.')

artifacts=[]
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file() and path.name!='11_OUTPUT_MANIFEST.csv':
        artifacts.append({'relative_path':str(path.relative_to(OUTPUT_ROOT)),'bytes':path.stat().st_size,'sha256':sha256_file(path)})
manifest=pd.DataFrame(artifacts); save_table(manifest,'11_OUTPUT_MANIFEST')
run={'experiment':'P1-E16.7H-2 frozen external compatibility, domain shift, calibration, and probability-quality audit',
     'status':'PASSED','completed_utc':datetime.now(timezone.utc).isoformat(),'endpoints':TASKS,
     'external_protocol':EXTERNAL_PROTOCOL,'methods':METHODS,'historically_untouched_claim':False,
     'evidence_label':'locked external re-evaluation','external_refit':False,'external_selection':False,
     'external_threshold_optimization':False,'external_recalibration':False,'external_adaptation':False,
     'bootstrap_repetitions':BOOTSTRAP_REPS,'performance_target_used':False,'integrity_passed':True,
     'environment':{'python':sys.version,'platform':platform.platform()}}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run,indent=2),encoding='utf-8')
print('P1-E16.7H-2 PASSED — EXTERNAL COMPATIBILITY, DOMAIN SHIFT, CALIBRATION, AND PROBABILITY-QUALITY AUDIT LOCKED')

## Interpretation guardrails

- AHEPA and Dryad/P-ADIC are separate external cohorts and must not be pooled.
- Balanced-accuracy chance is 0.50 for binary endpoints and 1/3 for the three-class endpoint.
- A positive Brier skill score means the model improves on the empirical-prevalence probability reference; a negative value means it is worse than that reference.
- Reliability curves and expected calibration error describe probability agreement; they are distinct from discrimination and threshold-dependent classification.
- Domain-classifier ROC-AUC and standardized mean differences diagnose dataset separability, not its cause.
